In [ ]:
import json
from collections import Counter
from datetime import datetime, timedelta
from itertools import product, zip_longest
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as goa
import regex
import requests
from plotly.colors import qualitative, sample_colorscale
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

from datetime import datetime
import pytz
from src.utils import (
    guardarExcel,
    guardarExcelMulti
)

from datetime import timedelta
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.api import getHistoricoMOW
from src.api.APIs import getInfoToposMSE, getInfoToposMSEWithoutCTC
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
from src.utils import (
    isEmpty,
    loadEstaciones,
    loadLocalizaciones,
    localizeFecha,
    parallelizeFunction,
    rellenarId,
    removeDoubleQuotes,
    splitDataframe,
    getEstacionamientos,
    loadEstacionSinCTC
    
)
from src.api.api import GraylogAPIProcessor
from src.utils.util import loadEstaciones,loadEstacionComercial
from src.api.APIs import getCirculacionesPlanificadas,getCirculacionesComerciales

In [ ]:
from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime
)
from src.processor import SitraProcessor, MIEProcessor, XPECProcessor


In [ ]:
# Tipos de tren que queremos
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "SUPRESIÓN",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ORIGEN",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    "Stopped": "STOP",
    "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "MANIOBRA_APROXIMACION"
            "MANIOBRA_LLEGADA",
            "MANIOBRA_SALIDA",
        ]
    )
}

In [ ]:

def cargarHistorico(
    start_date: str,
    end_date: str,
    estaciones: list[str],
    trenes: list[str],
    xSIV: bool = True,
    xSIVPLUS:bool = False,
    jCTC: bool = False,
    xREG: bool = False,
    pro: bool = True,
    maniobra:bool = True
):
    # Comprobamos que la fecha de fin sea después de la de inicio
    if end_date <= start_date:
        end_date = (pd.to_datetime(start_date) + timedelta(days=1)).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    historico = getHistoricoMOW(
        estaciones=estaciones,
        trenes=trenes,
        inicio=start_date,
        fin=end_date,
        xSIV=xSIV,
        xSIVPLUS = xSIVPLUS,
        jCTC=jCTC,
        xREG=xREG,
        pro=pro,
        maniobra= maniobra
    )
        
    if (xREG == False or JCTC == False): 
        historico = historico[
            (historico["Fecha"] >= pd.to_datetime(start_date))
            & (historico["Fecha"] <= pd.to_datetime(end_date))
        ]
    else:
         historico = historico[
            (historico["FechaHora"] >= pd.to_datetime(start_date))
            & (historico["FechaHora"] <= pd.to_datetime(end_date))
        ]
    # Usamos movimientos auditados
    # historico = historico[
    #     np.invert(historico["FuenteVía"].isin(["PLANNED", "SITRA_PROVIDED"]))
    # ]
    
    
    
    if (xREG == False): 
        historico = historico[historico["NTécnico"].apply(isValidCode)].dropna(
            subset=["Movimiento"]
        )
        historico["mov_ord"] = historico["Movimiento"].apply(mov_sorter.get)
    return historico


In [ ]:
start_date = "2026-04-15"
end_date = "2026-04-16"
estaciones = []

In [ ]:

estaciones = []
ntrenes = [rellenarId(el) for el in np.arange(100000)]
# ntrenes = [rellenarId(el) for el in np.arange(2000, 6000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=True,
    xSIVPLUS=False,
    jCTC=False,
    pro=True,
)
# historico_pro = historico_pro.sort_values(
#     by=["FechaOrigen", "NTécnico", "Fecha"]
# ).reset_index(drop=True)

# # Añadir información de la fecha
# historico_pro["Día"] = historico_pro["Fecha"].dt.date
# historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
# historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
# historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
historico_pro.columns

In [ ]:
df = historico_pro[historico_pro["FuenteMovimiento"] == "CTC_MIE"].copy()

In [ ]:
df_filter = df[df["Movimiento"].isin(["ORIGEN","LLEGADA"])].copy()

In [ ]:
df_filter.sort_values(by=["NTécnico", "Secuencia"]).reset_index(drop=True)

In [ ]:
sub_dfs = [fo for _, fo in df_filter.groupby("NTécnico")]

In [ ]:
df_planificada =  getCirculacionesPlanificadas("2026-04-15")

In [ ]:
for df in sub_dfs:
    df = df[df["Movimiento"].isin(["ORIGEN","LLEGADA"])]

In [ ]:
df_mse = pd.concat(sub_dfs).sort_values(by=["NTécnico", "Secuencia"]).reset_index(drop=True)

In [ ]:
df_planificada.sort_values(by=["NTécnico", "Secuencia"]).reset_index(drop=True)

In [ ]:
dict_planificadas = {tecnico: grupo for tecnico, grupo in df_planificada.groupby('NTécnico')}
dict_mse = {tecnico: grupo for tecnico, grupo in df_mse.groupby('NTécnico')}

In [ ]:
# Los que están en mse pero no en planificadas
df_solo_mse = df_mse[~df_mse["NTécnico"].isin(dict_planificadas.keys())]
# Los que están en planificadas pero no en mse
df_solo_planificadas = df_planificada[~df_planificada["NTécnico"].isin(dict_mse.keys())]

In [ ]:
df_solo_mse

In [ ]:
df_solo_mse = df_solo_mse[~df_solo_mse["Producto"].isin(["Material Vacio","Servicio Interno","Maquina Aislada","Transporte excepcional","Mercancias","T.L.E.","Maquina Aislada Mercancias","Mercancias RAM","Material vacio RAM"," "])].copy()

In [ ]:
df_solo_planificadas = df_solo_planificadas[~df_solo_planificadas["TipoTren"].isin([" ", "Mercancias", "T.L.E.", "Transporte excepcional", "Material Vacio", "Maquina Aislada Mercancias", "Servicio Interno", "Maquina Aislada", "Material vacio RAM", "MAQUINA AISLADA"])].copy()

In [ ]:
df_solo_planificadas.drop_duplicates(subset=["NTécnico"],keep='first', inplace=True)

In [ ]:
df_solo_planificadas

In [ ]:
# df_solo_planificadas.drop(columns=["Secuencia","Código","Vía_Planificada"], inplace=True)

In [ ]:
df_solo_mse.drop_duplicates(subset=["NTécnico"],keep='first', inplace=True)

In [ ]:
df_solo_mse.columns

In [ ]:
df_solo_mse = df_solo_mse[["NTécnico","CategoríaCirculación","Producto","Empresa","LíneaComercial","CódigoOrigen","NombreOrigen","CódigoDestino","NombreDestino"]]

In [ ]:
df_solo_mse

In [ ]:
df_solo_planificadas

In [ ]:
output_path = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\mse_planificadas.html")

In [ ]:
def generar_html(df_solo_mse, df_solo_planificadas, output_path, start_date=""):
    import datetime
    if not start_date:
        start_date = datetime.date.today().strftime("%d/%m/%Y")

    mse_cols = ["NTécnico", "Categoría", "Circulación", "Producto", "Empresa",
                "LíneaComercial", "CódigoOrigen", "NombreOrigen", "CódigoDestino", "NombreDestino"]
    plan_cols = ["NTécnico", "FechaOrigen", "Línea", "Compañia", "Operador", "TipoTren"]

    def df_to_chips(df, cols, prefix, accent_class):
        available = [c for c in cols if c in df.columns]
        detail_cols = [c for c in available if c != "NTécnico"]
        chips = ""
        for i, (_, row) in enumerate(df[available].iterrows()):
            ntecnico = row.get("NTécnico", "—")
            if ntecnico is None or (isinstance(ntecnico, float) and __import__('math').isnan(ntecnico)):
                ntecnico = "—"
            detail_fields = ""
            for c in detail_cols:
                val = row[c]
                if val is None or (isinstance(val, float) and __import__('math').isnan(val)) or val == "":
                    val_html = '<span class="empty-val">—</span>'
                else:
                    if c == "FechaOrigen":
                        try:
                            import pandas as pd
                            val = pd.to_datetime(val).strftime("%Y-%m-%d")
                        except Exception:
                            val = str(val)[:10]
                    val_html = f'<span class="detail-val">{val}</span>'
                detail_fields += f'<div class="detail-field"><span class="detail-key">{c}</span>{val_html}</div>'

            import json
            data_fields = json.dumps({"ntecnico": str(ntecnico), "fields": detail_fields})
            safe = data_fields.replace('"', '&quot;')

            chips += f"""<button class="nt-chip {accent_class}" style="animation-delay:{i*0.03:.2f}s" data-search="{ntecnico}" onclick="openModal({json.dumps(str(ntecnico)).replace('"', '&quot;')}, this)">{ntecnico}<div class="chip-detail-src" hidden>{detail_fields}</div></button>"""
        return chips

    import base64, os
    logo_path = "data/logo.png"
    if os.path.exists(logo_path):
        with open(logo_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")
        logo_tag = f'<img src="data:image/png;base64,{b64}" class="header-logo" alt="Logo">'
    else:
        logo_tag = '<div class="logo-placeholder">LOGO</div>'

    chips_mse  = df_to_chips(df_solo_mse,         mse_cols,  "mse",  "chip-dark")
    chips_plan = df_to_chips(df_solo_planificadas, plan_cols, "plan", "chip-mint")

    html = f"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Trenes sin coincidencia</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;600;700&family=IBM+Plex+Sans:ital,wght@0,300;0,400;0,500;0,600;1,400&display=swap" rel="stylesheet">
<style>
  *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}

  :root {{
    --bg:          #f4f7f4;
    --surface:     #ffffff;
    --surface2:    #eef4ee;
    --border:      #d4e4d4;
    --text:        #1a2e1a;
    --muted:       #6a8f6a;
    --green-dark:  #1c6b3a;
    --green-mid:   #2d9e57;
    --green-light: #e3f5e9;
    --green-mint:  #3dbf7a;
  }}

  body {{
    font-family: 'IBM Plex Sans', sans-serif;
    background: var(--bg);
    color: var(--text);
    min-height: 100vh;
  }}

  /* ══ HEADER ══ */
  .header {{
    background: #fff;
    border-bottom: 3px solid var(--green-dark);
    padding: 18px 32px;
    display: grid;
    grid-template-columns: 1fr auto 1fr;
    align-items: center;
    gap: 16px;
  }}
  .header-logo-col {{ display: flex; align-items: center; }}
  .header-logo {{ max-height: 64px; max-width: 180px; object-fit: contain; display: block; }}
  .logo-placeholder {{
    font-family: 'IBM Plex Sans', sans-serif; font-size: 13px; font-weight: 600;
    color: #ccc; border: 2px dashed #ddd; border-radius: 6px;
    padding: 10px 18px; letter-spacing: 0.06em;
  }}
  .header-center {{ text-align: center; display: flex; flex-direction: column; align-items: center; gap: 6px; }}
  .header-title {{
    font-family: 'Playfair Display', serif;
    font-size: 20px; font-weight: 700; color: #111; line-height: 1.2;
  }}
  .header-date-row {{ display: flex; align-items: center; gap: 8px; }}
  .header-date-label {{ font-size: 10px; letter-spacing: 0.14em; text-transform: uppercase; color: #999; font-weight: 500; }}
  .header-date-value {{
    font-size: 12px; font-weight: 600; color: var(--green-dark);
    background: var(--green-light); border: 1px solid var(--border);
    border-radius: 4px; padding: 2px 10px;
  }}
  .header-depts {{ display: flex; flex-direction: column; align-items: flex-end; gap: 4px; }}
  .dept-line {{ text-align: right; line-height: 1.5; }}
  .dept-line.dept-top  {{ font-size: 9.5px; color: #111; font-weight: 500; }}
  .dept-line.dept-main {{ font-size: 10px;  color: #111; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; }}

  /* ══ METRICS BAR ══ */
  .metrics-bar {{
    background: var(--green-dark); padding: 12px 32px;
    display: flex; gap: 16px; position: relative; overflow: hidden;
  }}
  .metrics-bar::before {{
    content: ''; position: absolute; inset: 0;
    background: repeating-linear-gradient(90deg, transparent, transparent 60px, rgba(255,255,255,0.04) 60px, rgba(255,255,255,0.04) 61px);
    pointer-events: none;
  }}
  .metric {{
    background: rgba(255,255,255,0.09); border: 1px solid rgba(255,255,255,0.14);
    border-radius: 8px; padding: 10px 18px; min-width: 140px;
    animation: fadeUp 0.5s ease both;
  }}
  .metric:nth-child(1) {{ animation-delay: 0.1s; border-top: 2px solid var(--green-mint); }}
  .metric:nth-child(2) {{ animation-delay: 0.2s; border-top: 2px solid rgba(255,255,255,0.4); }}
  .metric-label {{ font-size: 9px; letter-spacing: 0.14em; text-transform: uppercase; color: #000; margin-bottom: 4px; font-weight: 700; }}
  .metric-value {{ font-family: 'Playfair Display', serif; font-size: 28px; font-weight: 700; line-height: 1; color: #000; }}
  .metric:nth-child(1) .metric-value {{ color: #000; }}

  /* ══ CONTENT ══ */
  .content {{ padding: 36px 40px; display: flex; flex-direction: column; gap: 32px; }}

  .section {{ animation: fadeUp 0.5s ease both; }}
  .section:nth-child(1) {{ animation-delay: 0.2s; }}
  .section:nth-child(2) {{ animation-delay: 0.3s; }}

  .section-header {{ display: flex; align-items: center; gap: 12px; margin-bottom: 16px; }}
  .section-dot {{ width: 8px; height: 8px; border-radius: 50%; flex-shrink: 0; }}
  .dot-dark {{ background: var(--green-dark); box-shadow: 0 0 8px rgba(28,107,58,0.5); }}
  .dot-mint {{ background: var(--green-mint); box-shadow: 0 0 8px rgba(61,191,122,0.5); }}
  .section-title {{ font-family: 'Playfair Display', serif; font-size: 15px; font-weight: 600; color: var(--text); }}
  .section-count {{
    font-size: 11px; color: var(--muted); margin-left: auto;
    padding: 2px 10px; background: var(--surface);
    border: 1px solid var(--border); border-radius: 20px; font-weight: 500;
  }}

  /* Search */
  .search-box {{ position: relative; }}
  .search-icon {{ position: absolute; left: 10px; top: 50%; transform: translateY(-50%); color: var(--muted); font-size: 13px; pointer-events: none; }}
  .search-input {{
    background: var(--surface); border: 1px solid var(--border); border-radius: 6px;
    color: var(--text); font-family: 'IBM Plex Sans', sans-serif;
    font-size: 12px; padding: 7px 12px 7px 32px; outline: none; width: 200px;
    transition: border-color 0.2s, width 0.3s, box-shadow 0.2s;
  }}
  .search-input:focus {{ border-color: var(--green-mid); width: 260px; box-shadow: 0 0 0 3px rgba(45,158,87,0.12); }}

  /* ══ CHIPS PANEL ══ */
  .chips-panel {{
    background: var(--surface); border: 1px solid var(--border);
    border-radius: 10px; box-shadow: 0 2px 12px rgba(28,107,58,0.06);
    padding: 16px; display: flex; flex-wrap: wrap; gap: 8px;
  }}

  .nt-chip {{
    font-family: 'IBM Plex Sans', sans-serif;
    font-size: 12px; font-weight: 500; letter-spacing: 0.02em;
    padding: 5px 13px; border-radius: 4px; border: 1.5px solid;
    cursor: pointer; white-space: nowrap;
    opacity: 0; animation: chipIn 0.3s ease forwards;
    transition: background 0.18s, color 0.18s, border-color 0.18s,
                transform 0.15s, box-shadow 0.18s;
  }}
  .chip-dark {{
    color: var(--green-dark); background: var(--green-light); border-color: #b2d9bf;
  }}
  .chip-dark:hover {{
    background: var(--green-dark); color: #fff; border-color: var(--green-dark);
    box-shadow: 0 2px 10px rgba(28,107,58,0.25); transform: translateY(-1px);
  }}
  .chip-mint {{
    color: var(--green-mid); background: #edfaf3; border-color: #a8e8c5;
  }}
  .chip-mint:hover {{
    background: var(--green-mint); color: #fff; border-color: var(--green-mint);
    box-shadow: 0 2px 10px rgba(61,191,122,0.3); transform: translateY(-1px);
  }}

  .chip-detail-src {{ display: none; }}

  /* ══ EMPTY STATE ══ */
  .empty-state {{
    padding: 40px; text-align: center; color: var(--muted);
    font-size: 13px; background: var(--surface);
    border-radius: 10px; border: 1px solid var(--border);
  }}

  /* ══ MODAL OVERLAY ══ */
  .modal-overlay {{
    display: none;
    position: fixed; inset: 0; z-index: 100;
    background: rgba(15, 30, 15, 0.45);
    backdrop-filter: blur(3px);
    align-items: center; justify-content: center;
    padding: 24px;
  }}
  .modal-overlay.open {{ display: flex; animation: overlayIn 0.2s ease; }}

  .modal {{
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: 12px;
    box-shadow: 0 20px 60px rgba(0,0,0,0.18);
    width: 100%; max-width: 680px;
    max-height: 85vh; overflow-y: auto;
    animation: modalIn 0.22s ease;
  }}

  .modal-header {{
    display: flex; align-items: center; gap: 12px;
    padding: 18px 24px 14px;
    border-bottom: 1px solid var(--border);
    position: sticky; top: 0; background: var(--surface);
    border-radius: 12px 12px 0 0;
    z-index: 1;
  }}
  .modal-dot {{ width: 9px; height: 9px; border-radius: 50%; flex-shrink: 0; }}
  .modal-title {{
    font-family: 'Playfair Display', serif;
    font-size: 16px; font-weight: 700; color: var(--text);
  }}
  .modal-subtitle {{ font-size: 11px; color: var(--muted); margin-left: 2px; margin-top: 1px; }}
  .modal-close {{
    margin-left: auto;
    width: 28px; height: 28px; border-radius: 50%;
    border: 1px solid var(--border); background: var(--surface2);
    color: var(--muted); font-size: 16px; line-height: 1;
    cursor: pointer; display: flex; align-items: center; justify-content: center;
    transition: background 0.15s, color 0.15s;
    flex-shrink: 0;
  }}
  .modal-close:hover {{ background: #fde8e8; color: #c0392b; border-color: #f5c6c6; }}

  .modal-body {{ padding: 20px 24px 24px; }}

  .detail-grid {{
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(185px, 1fr));
    gap: 16px 28px;
  }}
  .detail-field {{ display: flex; flex-direction: column; gap: 4px; }}
  .detail-key {{
    font-size: 9px; letter-spacing: 0.14em;
    text-transform: uppercase; color: var(--muted); font-weight: 600;
  }}
  .detail-val {{ font-size: 13px; color: var(--text); font-weight: 500; }}
  .empty-val {{ font-size: 13px; color: var(--muted); font-style: italic; opacity: 0.5; }}

  /* ══ FOOTER ══ */
  .footer {{
    padding: 16px 32px; border-top: 1px solid var(--border);
    display: flex; align-items: center; justify-content: flex-end;
    background: var(--surface);
  }}
  .footer-text {{ font-size: 11px; color: var(--muted); letter-spacing: 0.06em; font-weight: 500; }}

  @keyframes fadeUp    {{ from {{ opacity:0; transform:translateY(12px); }} to {{ opacity:1; transform:translateY(0); }} }}
  @keyframes chipIn    {{ from {{ opacity:0; transform:scale(0.88); }}     to {{ opacity:1; transform:scale(1); }} }}
  @keyframes overlayIn {{ from {{ opacity:0; }} to {{ opacity:1; }} }}
  @keyframes modalIn   {{ from {{ opacity:0; transform:translateY(16px) scale(0.97); }} to {{ opacity:1; transform:translateY(0) scale(1); }} }}
  @keyframes pulse     {{ 0%,100% {{ opacity:1; }} 50% {{ opacity:0.35; }} }}

  ::-webkit-scrollbar {{ height: 5px; width: 5px; background: transparent; }}
  ::-webkit-scrollbar-thumb {{ background: var(--border); border-radius: 99px; }}
</style>
</head>
<body>

<!-- ══ HEADER ══ -->
<div class="header">
  <div class="header-logo-col">{logo_tag}</div>
  <div class="header-center">
    <div class="header-title">Circulaciones planificadas vs MSE</div>
    <div class="header-date-row">
      <span class="header-date-label">Fecha de dato</span>
      <span class="header-date-value">{start_date}</span>
    </div>
  </div>
  <div class="header-depts">
    <div class="dept-line dept-top">SD. de Sistemas y Medios Operacionales</div>
    <div class="dept-line dept-top">D. de Circulación y Gestión de Capacidad</div>
    <div class="dept-line dept-main">DG. de Operaciones y Explotación</div>
  </div>
</div>

<!-- ══ METRICS ══ -->
<div class="metrics-bar">
  <div class="metric">
    <div class="metric-label">Circulaciones rotuladas en MSE no planificadas</div>
    <div class="metric-value">{len(df_solo_mse)}</div>
  </div>
  <div class="metric">
    <div class="metric-label">Circulaciones planificadas no circuladas</div>
    <div class="metric-value">{len(df_solo_planificadas)}</div>
  </div>
</div>

<!-- ══ CONTENT ══ -->
<div class="content">

  <div class="section">
    <div class="section-header">
      <div class="section-dot dot-dark"></div>
      <div class="section-title">Circulaciones rotuladas en MSE no planificadas</div>
      <div class="section-count">{len(df_solo_mse)} registros</div>
      <div class="search-box">
        <span class="search-icon">⌕</span>
        <input class="search-input" type="text" placeholder="Filtrar NTécnico..."
               oninput="filterChips(this, 'chips-mse')">
      </div>
    </div>
    {'<div class="chips-panel" id="chips-mse">' + chips_mse + '</div>' if len(df_solo_mse) > 0 else '<div class="empty-state">No hay registros</div>'}
  </div>

  <div class="section">
    <div class="section-header">
      <div class="section-dot dot-mint"></div>
      <div class="section-title">Circulaciones planificadas no circuladas</div>
      <div class="section-count">{len(df_solo_planificadas)} registros</div>
      <div class="search-box">
        <span class="search-icon">⌕</span>
        <input class="search-input" type="text" placeholder="Filtrar NTécnico..."
               oninput="filterChips(this, 'chips-plan')">
      </div>
    </div>
    {'<div class="chips-panel" id="chips-plan">' + chips_plan + '</div>' if len(df_solo_planificadas) > 0 else '<div class="empty-state">No hay registros</div>'}
  </div>

</div>

<!-- ══ FOOTER ══ -->
<div class="footer">
  <span class="footer-text" id="ts"></span>
</div>

<!-- ══ MODAL ══ -->
<div class="modal-overlay" id="modal-overlay" onclick="handleOverlayClick(event)">
  <div class="modal" id="modal-box">
    <div class="modal-header">
      <div class="modal-dot" id="modal-dot"></div>
      <div>
        <div class="modal-title" id="modal-title"></div>
        <div class="modal-subtitle">Detalle del tren</div>
      </div>
      <button class="modal-close" onclick="closeModal()">✕</button>
    </div>
    <div class="modal-body">
      <div class="detail-grid" id="modal-body"></div>
    </div>
  </div>
</div>

<script>
  document.getElementById('ts').textContent = new Date().toLocaleString('es-ES', {{
    day:'2-digit', month:'short', year:'numeric', hour:'2-digit', minute:'2-digit'
  }});

  function openModal(ntecnico, chipEl) {{
    const isDark = chipEl.classList.contains('chip-dark');
    const detailHTML = chipEl.querySelector('.chip-detail-src').innerHTML;

    document.getElementById('modal-title').textContent = ntecnico;
    document.getElementById('modal-body').innerHTML = detailHTML;
    document.getElementById('modal-dot').style.background = isDark
      ? 'var(--green-dark)' : 'var(--green-mint)';

    const overlay = document.getElementById('modal-overlay');
    overlay.classList.add('open');
    document.body.style.overflow = 'hidden';
  }}

  function closeModal() {{
    document.getElementById('modal-overlay').classList.remove('open');
    document.body.style.overflow = '';
  }}

  function handleOverlayClick(e) {{
    if (e.target === document.getElementById('modal-overlay')) closeModal();
  }}

  document.addEventListener('keydown', e => {{ if (e.key === 'Escape') closeModal(); }});

  function filterChips(input, chipsId) {{
    const q = input.value.toLowerCase();
    document.getElementById(chipsId).querySelectorAll('.nt-chip').forEach(chip => {{
      chip.style.display = chip.dataset.search.toLowerCase().includes(q) ? '' : 'none';
    }});
  }}
</script>
</body>
</html>"""

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"HTML generado en: {output_path}")

In [ ]:



generar_html(df_solo_mse, df_solo_planificadas,output_path,start_date=start_date)